<a href="https://colab.research.google.com/github/BidishaSETH/Deformable-TouchSDF-Simulation/blob/main/Deformable_TouchSDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch torchvision numpy pybullet matplotlib

  Using cached pybullet-3.2.7.tar.gz (80.5 MB)
  Preparing metadata (setup.py) ... done
  Created wheel for pybullet: filename=pybullet-3.2.7-cp313-cp313-linux_x86_64.whl size=99120539 sha256=1b454cfbb858df4e458433dd5b66f82d62efbd59535e162e23df5982b07935df
  Stored in directory: /root/.cache/pip/wheels/8a/f0/6a/3f626b4ee04f8ec61e40021b095adf5359f9bd426065a368ac
Successfully built pybullet


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

# Define the updated dataset including nominal shape codes (latent vectors)
class VariableStiffnessWithLatentDataset(Dataset):
    def __init__(self, num_samples=1000, latent_dim=128, min_force=0.0, max_force=15.0, min_stiffness=10.0, max_stiffness=200.0):
        # 3D spatial coordinates
        self.x = torch.empty(num_samples, 3).uniform_(-0.05, 0.05)

        # Generate random 128-dimensional latent shape codes for each sample
        # In standard setups, this represents different object geometries
        self.latent_codes = torch.randn(num_samples, latent_dim) * 0.1

        # Physical parameter sampling
        self.stiffness = torch.empty(num_samples, 1).uniform_(min_stiffness, max_stiffness)
        self.forces = torch.empty(num_samples, 1).uniform_(min_force, max_force)

        # Contact depth and nominal shape ground-truth SDF
        self.tactile_depth = self.forces / self.stiffness
        self.s_gt = -self.tactile_depth

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return (
            self.x[idx],
            self.latent_codes[idx],
            self.forces[idx],
            self.stiffness[idx],
            self.s_gt[idx],
            self.tactile_depth[idx]
        )

# Instantiate the dataset
latent_dim = 128
dataset_with_latents = VariableStiffnessWithLatentDataset(num_samples=1000, latent_dim=latent_dim)

# Build dictionary for DataFrame conversion
data_dict = {
    'x': dataset_with_latents.x[:, 0].numpy(),
    'y': dataset_with_latents.x[:, 1].numpy(),
    'z': dataset_with_latents.x[:, 2].numpy(),
    'force_N': dataset_with_latents.forces.squeeze().numpy(),
    'stiffness_N_per_m': dataset_with_latents.stiffness.squeeze().numpy(),
    'ground_truth_sdf': dataset_with_latents.s_gt.squeeze().numpy(),
    'tactile_depth_m': dataset_with_latents.tactile_depth.squeeze().numpy()
}

# Adding latent shape vector features to the DataFrame columns (latent_0 to latent_127)
for i in range(latent_dim):
    data_dict[f'latent_{i}'] = dataset_with_latents.latent_codes[:, i].numpy()

# Convert to DataFrame and save to CSV
df_latents = pd.DataFrame(data_dict)
output_filename = 'synthetic_tactile_dataset_with_latents.csv'
df_latents.to_csv(output_filename, index=False)

display(df_latents.head())
print(f"\nDataset saved as {output_filename}")
print(f"DataFrame Shape: {df_latents.shape} (Includes {latent_dim} latent features)")

,x,y,z,force_N,stiffness_N_per_m,ground_truth_sdf,tactile_depth_m,latent_0,latent_1,latent_2,...,latent_118,latent_119,latent_120,latent_121,latent_122,latent_123,latent_124,latent_125,latent_126,latent_127
0,-0.024860,0.038078,0.038525,13.578076,136.153290,-0.099726,0.099726,0.044952,-0.075678,-0.055138,...,-0.032782,-0.002212,-0.020742,0.077762,-0.032950,-0.133435,0.042345,0.092247,-0.051742,0.005828
1,-0.002292,-0.049937,-0.028781,12.061390,161.203918,-0.074821,0.074821,0.036371,0.088000,-0.092463,...,0.014605,-0.125454,-0.060117,0.088971,0.098488,0.019258,0.091207,-0.149947,-0.269717,0.005608
2,0.010021,-0.031197,0.019760,2.798110,167.989624,-0.016656,0.016656,-0.016211,0.236255,-0.020988,...,0.045995,-0.102033,0.008703,0.007100,-0.071439,-0.148022,-0.127791,0.020357,-0.105046,-0.018854
3,-0.044107,-0.020357,-0.038298,10.420947,183.345169,-0.056838,0.056838,0.013682,0.129344,-0.033682,...,-0.067762,-0.092069,0.028667,-0.004823,-0.049158,0.100068,0.120750,-0.037288,-0.005911,0.031115
4,0.047192,0.040045,-0.009060,0.075684,113.995918,-0.000664,0.000664,-0.082431,0.086764,-0.055264,...,-0.007474,0.064910,-0.019259,0.064821,-0.027715,-0.275245,0.077591,-0.031172,0.313480,-0.101564



Dataset saved as synthetic_tactile_dataset_with_latents.csv
DataFrame Shape: (1000, 135) (Includes 128 latent features)


In [ ]:
# Model Architecture
class DeformableTouchSDF(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()

        # Branch 1: Base Shape MLP (Predicts un-dented nominal shape S0)
        self.base_mlp = nn.Sequential(
            nn.Linear(3 + latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

        # Branch 2: Offset Subnet (Predicts dent depth using Force and Compliance)
        self.offset_net = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Softplus() # Keeps indentation depth strictly positive
        )

    def forward(self, x, latent_code, force, stiffness):
        # Calculate material compliance m = 1 / k
        compliance = 1.0 / (stiffness + 1e-6)

        # 1. Predict nominal shape
        base_input = torch.cat([x, latent_code], dim=-1)
        s_0 = self.base_mlp(base_input)

        # 2. Predict surface indentation displacement offset
        offset_input = torch.cat([force, compliance], dim=-1)
        delta_d = self.offset_net(offset_input)

        # 3. Deformed SDF = Nominal Shape - Indent Dent
        s_deformed = s_0 - delta_d
        return s_deformed, s_0, delta_d

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DeformableTouchSDF(latent_dim=128).to(device)

print("DeformableTouchSDF initialized successfully!")
print("Device:", device)
print("Trainable parameters:",
      sum(p.numel() for p in model.parameters()))

DeformableTouchSDF initialized successfully!
Device: cpu
Trainable parameters: 100098


In [ ]:
class DeformableSDFLoss(nn.Module):

    def __init__(
        self,
        lambda_nominal=1.0,
        lambda_contact=0.1,
        lambda_physics=0.01
    ):
        super().__init__()

        self.lambda_nominal = lambda_nominal
        self.lambda_contact = lambda_contact
        self.lambda_physics = lambda_physics

        self.l1_loss = nn.L1Loss()
        self.mse_loss = nn.MSELoss()

    def forward(
        self,
        s_deformed_pred,
        s_0_pred,
        s_gt,
        delta_d_pred,
        force,
        stiffness,
        tactile_depth_gt
    ):

        # Ground-truth deformed SDF
        s_deformed_gt = s_gt - tactile_depth_gt

        # 1. Deformed geometry reconstruction
        l_sdf = self.l1_loss(
            s_deformed_pred,
            s_deformed_gt
        )

        # 2. Nominal shape restoration
        l_nominal = self.l1_loss(
            s_0_pred,
            s_gt
        )

        # 3. Contact indentation loss
        l_contact = self.mse_loss(
            delta_d_pred,
            tactile_depth_gt
        )

        # 4. Physical elasticity loss
        expected_displacement = force / (stiffness + 1e-6)

        l_physics = self.mse_loss(
            delta_d_pred,
            expected_displacement
        )

        total_loss = (
            l_sdf
            + self.lambda_nominal * l_nominal
            + self.lambda_contact * l_contact
            + self.lambda_physics * l_physics
        )

        return total_loss
        loss_function = DeformableSDFLoss().to(device)


In [ ]:
# Training
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

EPOCHS = 250
LATENT_DIM = 128
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Ensure `model` exists (uses `trained_model` or `model` from globals, or builds a fallback ScaledOffsetNet)
if "model" not in globals():
    if "trained_model" in globals():
        model = trained_model
    else:
        class ScaledOffsetNet(nn.Module):
            def __init__(self, latent_dim=128):
                super().__init__()
                self.base_mlp = nn.Sequential(
                    nn.Linear(3 + latent_dim, 128), nn.ReLU(),
                    nn.Linear(128, 128), nn.ReLU(),
                    nn.Linear(128, 1),
                )
                self.delta_mlp = nn.Sequential(
                    nn.Linear(2, 64), nn.ReLU(),
                    nn.Linear(64, 64), nn.ReLU(),
                    nn.Linear(64, 1), nn.Softplus(),
                )

            def forward(self, x, latent_code, force, stiffness):
                s_0 = self.base_mlp(torch.cat([x, latent_code], dim=-1))
                delta_d = self.delta_mlp(torch.cat([force, stiffness], dim=-1))
                s_deformed = s_0 - delta_d
                return s_deformed, s_0, delta_d

        model = ScaledOffsetNet(latent_dim=LATENT_DIM)

model = model.to(device)

# Ensure `var_dataloader` exists (finds any existing DataLoader in memory or builds a 1000-sample dataset)
if "var_dataloader" not in globals():
    existing_loaders = [v for v in globals().values() if isinstance(v, DataLoader)]
    if existing_loaders:
        var_dataloader = existing_loaders[0]
    else:
        torch.manual_seed(42)
        n_samples = 1000
        x_data = (torch.rand(n_samples, 3) - 0.5) * 0.1
        f_data = torch.empty(n_samples, 1).uniform_(0.0, 15.0)
        k_data = torch.empty(n_samples, 1).uniform_(10.0, 200.0)
        depth_data = f_data / k_data
        s_gt_data = -depth_data
        var_dataloader = DataLoader(
            TensorDataset(x_data, f_data, k_data, s_gt_data, depth_data),
            batch_size=64,
            shuffle=True,
        )

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

def loss_function(s_deformed_pred, s_0_pred, s_gt, delta_d_pred, force, stiffness, tactile_depth):
    l_sdf = F.mse_loss(s_deformed_pred, s_gt)
    l_nominal = F.l1_loss(s_0_pred, torch.zeros_like(s_0_pred))
    l_contact = F.mse_loss(delta_d_pred, tactile_depth)
    l_physics = F.mse_loss(delta_d_pred * stiffness, force)

    total_loss = l_sdf + (5.0 * l_nominal) + l_contact + (0.01 * l_physics)
    return total_loss, l_sdf, l_nominal, l_contact, l_physics

training_losses, sdf_losses, nominal_losses, contact_losses, physics_losses = [], [], [], [], []

print("\n" + "=" * 65)
print("STARTING DEFORMABLE TOUCHSDF TRAINING (LOW SHAPE ERROR)")
print("=" * 65)

for epoch in range(EPOCHS):
    model.train()
    running_total = running_sdf = running_nominal = running_contact = running_physics = 0.0

    for batch in var_dataloader:
        x, force, stiffness, s_gt, tactile_depth = [b.to(device) for b in batch]
        latent_code = torch.zeros((x.size(0), LATENT_DIM), device=device)

        optimizer.zero_grad()

        s_deformed_pred, s_0_pred, delta_d_pred = model(x, latent_code, force, stiffness)
        total_loss, l_sdf, l_nominal, l_contact, l_physics = loss_function(
            s_deformed_pred, s_0_pred, s_gt, delta_d_pred, force, stiffness, tactile_depth
        )

        total_loss.backward()
        optimizer.step()

        running_total += total_loss.item()
        running_sdf += l_sdf.item()
        running_nominal += l_nominal.item()
        running_contact += l_contact.item()
        running_physics += l_physics.item()

    scheduler.step()

    n_batches = len(var_dataloader)
    avg_total, avg_sdf, avg_nominal, avg_contact, avg_physics = (
        running_total / n_batches, running_sdf / n_batches,
        running_nominal / n_batches, running_contact / n_batches, running_physics / n_batches
    )

    training_losses.append(avg_total)
    sdf_losses.append(avg_sdf)
    nominal_losses.append(avg_nominal)
    contact_losses.append(avg_contact)
    physics_losses.append(avg_physics)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(
            f"Epoch [{epoch+1:03d}/{EPOCHS}] | "
            f"Total: {avg_total:.3f} | SDF: {avg_sdf:.3f} | "
            f"Nominal: {avg_nominal:.3f} | Contact: {avg_contact:.3f} | Physics: {avg_physics:.3f}"
        )

print("\n" + "=" * 65)
print("250-EPOCH TRAINING COMPLETED SUCCESSFULLY!")
print("=" * 65)

trained_model = model
torch.save(model.state_dict(), "deformable_touchsdf_250_epochs.pth")



STARTING DEFORMABLE TOUCHSDF TRAINING (LOW SHAPE ERROR)
Epoch [001/250] | Total: 0.008 | SDF: 0.000 | Nominal: 0.001 | Contact: 0.000 | Physics: 0.140
Epoch [010/250] | Total: 0.004 | SDF: 0.000 | Nominal: 0.001 | Contact: 0.000 | Physics: 0.001
Epoch [020/250] | Total: 0.003 | SDF: 0.000 | Nominal: 0.001 | Contact: 0.000 | Physics: 0.001
Epoch [030/250] | Total: 0.003 | SDF: 0.000 | Nominal: 0.001 | Contact: 0.000 | Physics: 0.001
Epoch [040/250] | Total: 0.003 | SDF: 0.000 | Nominal: 0.001 | Contact: 0.000 | Physics: 0.001
Epoch [050/250] | Total: 0.002 | SDF: 0.000 | Nominal: 0.000 | Contact: 0.000 | Physics: 0.001
Epoch [060/250] | Total: 0.002 | SDF: 0.000 | Nominal: 0.000 | Contact: 0.000 | Physics: 0.001
Epoch [070/250] | Total: 0.001 | SDF: 0.000 | Nominal: 0.000 | Contact: 0.000 | Physics: 0.001
Epoch [080/250] | Total: 0.001 | SDF: 0.000 | Nominal: 0.000 | Contact: 0.000 | Physics: 0.001
Epoch [090/250] | Total: 0.001 | SDF: 0.000 | Nominal: 0.000 | Contact: 0.000 | Physics:

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

dev = next(trained_model.parameters()).device

# 1. Upgrading offset_net with normalized inputs & 2 hidden layers so it doesn't saturate at 15 N
class ScaledOffsetNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 128),
            nn.GELU(),
            nn.Linear(128, 128),
            nn.GELU(),
            nn.Linear(128, 1),
            nn.Softplus()
        )

    def forward(self, x):
        # x is [force, compliance]; normalizing both to [0, 1] scale
        f_norm = x[:, 0:1] / 15.0
        c_norm = x[:, 1:2] / 0.1
        return self.net(torch.cat([f_norm, c_norm], dim=-1))

trained_model.offset_net = ScaledOffsetNet().to(dev)

# 2. Training offset_net to sub-millimeter precision across Force [0, 15 N] and Stiffness [10, 200 N/m]
n_samples = 8000
k_train = torch.empty(n_samples, 1, device=dev).uniform_(10.0, 200.0)
f_train = torch.empty(n_samples, 1, device=dev).uniform_(0.0, 15.0)
f_train[:int(0.15 * n_samples)] = 0.0  # Exact zero-force anchors
d_gt_mm = (f_train / k_train) * 1000.0

optimizer = torch.optim.AdamW(trained_model.offset_net.parameters(), lr=3e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=500, eta_min=1e-6)

trained_model.train()
batch_size = 256
for epoch in range(500):
    perm = torch.randperm(n_samples, device=dev)
    for i in range(0, n_samples, batch_size):
        idx = perm[i:i + batch_size]
        optimizer.zero_grad()

        comp = 1.0 / (k_train[idx] + 1e-6)
        pred_mm = trained_model.offset_net(torch.cat([f_train[idx], comp], dim=-1)) * 1000.0

        # Relative + absolute millimeter loss ensures high accuracy at both small and peak (15 N) loads
        loss = F.smooth_l1_loss(pred_mm, d_gt_mm[idx], beta=0.2)
        loss.backward()
        optimizer.step()
    scheduler.step()

trained_model.eval()
torch.save(trained_model.state_dict(), "deformable_touchsdf_250_epochs.pth")
print("Scaled offset_net trained!")


Scaled offset_net trained!


In [ ]:
import numpy as np
import torch

def evaluate_deformable_sdf(f, k, s_def):
    """f [N], k [N/m], model outputs in metres (1-D numpy). Returns mean reconstruction error (mm)."""
    mm = 1e3
    d = f / k
    e_def = s_def + d
    mean_reconstructed_error_mm = mm * np.abs(e_def).mean()
    return mean_reconstructed_error_mm

# ----- safe thin torch wrapper -----
net = globals().get("trained_model") or globals().get("model")
if net is None:
    raise RuntimeError("Neither `trained_model` nor `model` is defined in memory.")

net.eval()
dev = next(net.parameters()).device
LAT = net.base_mlp[0].in_features - 3

@torch.no_grad()
def run_model(f, k, seed=0):
    n = len(f)
    g = torch.Generator().manual_seed(seed)
    x = ((torch.rand(n, 3, generator=g) - 0.5) * 0.1).to(dev)
    t = lambda a: torch.as_tensor(np.asarray(a), dtype=torch.float32, device=dev).reshape(-1, 1)
    out = net(x, torch.zeros(n, LAT, device=dev), t(f), t(k))
    return [a.cpu().numpy().ravel().astype(np.float64) for a in out]

rng = np.random.default_rng(2024)
N = 5000
f_te, k_te = rng.uniform(0, 15, N), rng.uniform(10, 200, N)
s_def, _, _ = run_model(f_te, k_te)

mean_reconstructed_error = evaluate_deformable_sdf(f_te, k_te, s_def)
print(f"Model Evaluated Successfully! Mean Reconstruction Error: {mean_reconstructed_error:.3f} mm")


Model Evaluated Successfully! Mean Reconstruction Error: 0.220 mm
